# Great Basin post-fire annual herbaceous cover

Shared working notebook for the class project.

**Question:** do burned areas in the Great Basin show greater post-fire increases in annual herbaceous cover, as a proxy for invasive annual grass expansion, compared with unburned areas nearby?

This notebook walks through the full setup end-to-end so anyone cloning the repo can reproduce it from raw downloads:

1. **Setup** — imports, paths, project CRS
2. **Great Basin boundary** — load and reproject to the project CRS
3. **MTBS preprocessing** — load, clean attributes, reproject, filter to fires inside the Great Basin
4. **RCMAP preprocessing** — clip each annual raster to the Great Basin polygon (writes cleaned files to `data/processed/rcmap/`)
5. **Overview map** — study area with fire perimeters
6. **Worked example** — one fire: pre- vs post-fire mean annual herbaceous cover

See `ANALYSIS_PLAN.md` for next steps.

## 1. Setup

We work in **EPSG:5070** (CONUS Albers Equal Area) as the project CRS, because RCMAP is published in that CRS and equal-area projections are appropriate for area-based summaries.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask

DATA = Path('data')

BOUNDARY = DATA / 'raw' / 'boundaries' / 'great_basin_huc2_16_wbd.geojson'
MTBS_RAW = DATA / 'raw' / 'mtbs' / 'mtbs_perims_DD.shp'
RCMAP_RAW_DIR = DATA / 'raw' / 'rcmap'

MTBS_CLEAN = DATA / 'processed' / 'mtbs' / 'mtbs_great_basin_cleaned.gpkg'
RCMAP_CLIPPED_DIR = DATA / 'processed' / 'rcmap'

MTBS_CLEAN.parent.mkdir(parents=True, exist_ok=True)
RCMAP_CLIPPED_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_CRS = 'EPSG:5070'
RCMAP_YEARS = range(2012, 2021)
RCMAP_NODATA = 101

## 2. Great Basin boundary

Load and reproject to the project CRS.

In [ ]:
gb = gpd.read_file(BOUNDARY).to_crs(PROJECT_CRS)
print('Great Basin polygon:', len(gb), 'feature(s), CRS =', gb.crs)
gb.head()

## 3. MTBS preprocessing

The raw MTBS shapefile is nationwide. We:

1. Load it
2. Parse `Ig_Date` into a real datetime and derive `IgnitionYear`
3. Drop rows missing critical attributes (ignition date, geometry, acreage)
4. Reproject to the project CRS
5. Spatial-join to keep only fires intersecting the Great Basin polygon
6. Write the cleaned subset to `data/processed/mtbs/` as a GeoPackage

In [ ]:
mtbs = gpd.read_file(MTBS_RAW)
print('MTBS raw:', len(mtbs), 'rows, CRS =', mtbs.crs)

mtbs['Ig_Date'] = pd.to_datetime(mtbs['Ig_Date'], errors='coerce')
mtbs['IgnitionYear'] = mtbs['Ig_Date'].dt.year

before = len(mtbs)
mtbs = mtbs.dropna(subset=['Ig_Date', 'geometry', 'BurnBndAc']).copy()
mtbs = mtbs[mtbs.geometry.is_valid & ~mtbs.geometry.is_empty]
print(f'Dropped {before - len(mtbs)} rows with missing/invalid attributes or geometry')

mtbs = mtbs.to_crs(PROJECT_CRS)
print('MTBS reprojected to', mtbs.crs)

In [ ]:
fires = gpd.sjoin(mtbs, gb[['geometry']], predicate='intersects', how='inner')
fires = fires.drop(columns=['index_right']).reset_index(drop=True)
print(f'Fires intersecting the Great Basin: {len(fires)}')

fires.to_file(MTBS_CLEAN, driver='GPKG')
print('Wrote cleaned MTBS to', MTBS_CLEAN)
fires[['Event_ID', 'Incid_Name', 'Ig_Date', 'IgnitionYear', 'BurnBndAc']].head()

## 4. RCMAP preprocessing — clip to Great Basin

RCMAP rasters are published as CONUS-wide mosaics. We clip each annual raster to the Great Basin polygon and write the result to `data/processed/rcmap/`. This keeps the pre/post extraction step fast.

This cell is **idempotent**: if the clipped file already exists, it is skipped.

In [ ]:
def clip_rcmap_year(year):
    src_path = RCMAP_RAW_DIR / f'rcmap_annual_herbaceous_{year}.tif'
    dst_path = RCMAP_CLIPPED_DIR / f'rcmap_annual_herbaceous_{year}_great_basin_clipped.tif'
    if dst_path.exists():
        return dst_path, 'skipped (exists)'
    if not src_path.exists():
        return dst_path, f'missing raw: {src_path}'
    with rasterio.open(src_path) as src:
        gb_in_src = gb.to_crs(src.crs)
        arr, transform = mask(src, gb_in_src.geometry, crop=True, filled=True, nodata=RCMAP_NODATA)
        profile = src.profile.copy()
        profile.update(height=arr.shape[1], width=arr.shape[2], transform=transform, nodata=RCMAP_NODATA)
    with rasterio.open(dst_path, 'w', **profile) as dst:
        dst.write(arr)
    return dst_path, 'written'

for yr in RCMAP_YEARS:
    path, status = clip_rcmap_year(yr)
    print(f'{yr}: {status:20s} -> {path.name}')

In [ ]:
sample_year = 2016
sample_path = RCMAP_CLIPPED_DIR / f'rcmap_annual_herbaceous_{sample_year}_great_basin_clipped.tif'
with rasterio.open(sample_path) as src:
    print('CRS        :', src.crs)
    print('Shape      :', src.height, 'x', src.width)
    print('Resolution :', src.res)
    print('Dtype      :', src.dtypes[0])
    print('Nodata     :', src.nodata)

## 5. Overview map

Great Basin polygon with fire perimeters overlaid.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
gb.boundary.plot(ax=ax, color='black', linewidth=1)
fires.plot(ax=ax, color='red', alpha=0.5, edgecolor='none')
ax.set_title(f'MTBS fires intersecting the Great Basin (n={len(fires)})')
ax.set_xlabel('x (m, EPSG:5070)'); ax.set_ylabel('y (m, EPSG:5070)')
plt.tight_layout()
plt.show()

## 6. Worked example: one fire, pre- and post-fire cover

Pick one fire whose ignition year has at least one RCMAP year before and after, clip RCMAP to the perimeter, and compute the mean annual herbaceous cover in each window.

In [ ]:
eligible = fires[(fires['IgnitionYear'] >= 2013) & (fires['IgnitionYear'] <= 2019)].copy()
print('Fires eligible for 1-year pre/post with 2012–2020 RCMAP:', len(eligible))

example = eligible.sort_values('BurnBndAc', ascending=False).iloc[0]
print('\nExample fire:')
print(' Name :', example['Incid_Name'])
print(' Year :', int(example['IgnitionYear']))
print(' Acres:', int(example['BurnBndAc']))

In [ ]:
def zonal_mean(raster_path, geom, geom_crs):
    """Mean raster value inside geom, ignoring RCMAP nodata (101)."""
    with rasterio.open(raster_path) as src:
        geom_proj = gpd.GeoSeries([geom], crs=geom_crs).to_crs(src.crs).iloc[0]
        arr, _ = mask(src, [geom_proj], crop=True, filled=True, nodata=RCMAP_NODATA)
    arr = arr[0]
    valid = arr[arr != RCMAP_NODATA]
    if valid.size == 0:
        return np.nan, 0
    return float(valid.mean()), int(valid.size)

year = int(example['IgnitionYear'])
pre_path = RCMAP_CLIPPED_DIR / f'rcmap_annual_herbaceous_{year - 1}_great_basin_clipped.tif'
post_path = RCMAP_CLIPPED_DIR / f'rcmap_annual_herbaceous_{year + 1}_great_basin_clipped.tif'

pre_mean, pre_n = zonal_mean(pre_path, example.geometry, fires.crs)
post_mean, post_n = zonal_mean(post_path, example.geometry, fires.crs)

print(f'Pre-fire  ({year - 1}): mean = {pre_mean:.2f}%  (n pixels = {pre_n})')
print(f'Post-fire ({year + 1}): mean = {post_mean:.2f}%  (n pixels = {post_n})')
print(f'Change              : {post_mean - pre_mean:+.2f} percentage points')

## Phase 2: Burned-area summary

Scale the single-fire extraction to all eligible fires. For each fire with ignition year 2013–2019, compute pre-fire (year before) and post-fire (year after) mean annual herbaceous cover.

### Helper functions and parameters

In [ ]:
from contextlib import ExitStack
from shapely import make_valid
from shapely.geometry import mapping

ELIGIBLE_IGNITION_YEARS = list(range(2013, 2020))

def rcmap_clipped_path(year):
    return RCMAP_CLIPPED_DIR / f"rcmap_annual_herbaceous_{year}_great_basin_clipped.tif"

def zonal_mean_improved(src, geom):
    """Extract mean raster value inside geometry, excluding nodata."""
    try:
        data, _ = mask(src, [mapping(geom)], crop=True,
                       nodata=src.nodata, filled=True, all_touched=False)
    except ValueError:
        return np.nan, 0
    band = data[0].astype("float32")
    valid = np.isfinite(band)
    if src.nodata is not None:
        valid &= band != src.nodata
    n = int(valid.sum())
    if n == 0:
        return np.nan, 0
    return float(band[valid].mean()), n

### Filter fires to eligible ignition years

In [ ]:
# Reload the cleaned fires
fires_all = gpd.read_file(MTBS_CLEAN)

# Filter to eligible ignition years (need pre and post RCMAP data)
fires_eligible = fires_all[fires_all["IgnitionYear"].isin(ELIGIBLE_IGNITION_YEARS)].copy()
fires_eligible["PreFireYear"]  = fires_eligible["IgnitionYear"] - 1
fires_eligible["PostFireYear"] = fires_eligible["IgnitionYear"] + 1
fires_eligible["BurnedAreaSqKm"] = fires_eligible.geometry.area / 1_000_000

print(f"Total fires in cleaned MTBS: {len(fires_all)}")
print(f"Eligible fires (ignition 2013-2019): {len(fires_eligible)}")
print("\nFires by ignition year:")
print(fires_eligible["IgnitionYear"].value_counts().sort_index())

### Extract pre/post RCMAP means for all fires

Opens all RCMAP rasters once and loops over eligible fires. This may take a few minutes.

In [ ]:
records = []
with ExitStack() as stack:
    # Open all RCMAP years at once
    rasters = {y: stack.enter_context(rasterio.open(rcmap_clipped_path(y))) 
               for y in RCMAP_YEARS}
    
    # Loop over each eligible fire
    for idx, row in fires_eligible.iterrows():
        pre_y  = int(row.PreFireYear)
        post_y = int(row.PostFireYear)
        
        pre_mean,  pre_n  = zonal_mean_improved(rasters[pre_y],  row.geometry)
        post_mean, post_n = zonal_mean_improved(rasters[post_y], row.geometry)
        
        # Skip fires with no valid data
        if np.isnan(pre_mean) or np.isnan(post_mean):
            continue
        
        records.append({
            "Event_ID":       row.Event_ID,
            "Incid_Name":     row.Incid_Name,
            "IgnitionYear":   int(row.IgnitionYear),
            "PreFireYear":    pre_y,
            "PostFireYear":   post_y,
            "BurnBndAc":      row.BurnBndAc,
            "BurnedAreaSqKm": row.BurnedAreaSqKm,
            "PreMean":        pre_mean,
            "PostMean":       post_mean,
            "AbsChange":      post_mean - pre_mean,
            "PrePixels":      pre_n,
            "PostPixels":     post_n,
        })

pre_post = pd.DataFrame(records).sort_values(["IgnitionYear","Event_ID"]).reset_index(drop=True)
print(f"Fires with valid pre + post means: {len(pre_post)}")
pre_post.head(10)

### Phase 2 Summary Statistics

In [ ]:
print(f"Fires analysed: {len(pre_post)}")
print(f"Mean pre-fire cover : {pre_post['PreMean'].mean():.2f}%")
print(f"Mean post-fire cover: {pre_post['PostMean'].mean():.2f}%")
print(f"Mean absolute change: {pre_post['AbsChange'].mean():+.2f} percentage points")
print(f"Fires with increase : {(pre_post['AbsChange'] > 0).sum()}")
print(f"Fires with decrease : {(pre_post['AbsChange'] < 0).sum()}")
print("\nDescriptive statistics:")
pre_post[['PreMean','PostMean','AbsChange']].describe().round(2)

## Next steps

Phase 2 complete! See `ANALYSIS_PLAN.md` for remaining phases:

- Phase 3: Burned vs unburned comparison with control polygons
- Phase 4: Stratification by fire year, size, severity
- Phase 5: Robustness checks and sensitivity analysis